# Treino do Qwen do MixAI (Colab)
Siga em ordem: **1.** Ambiente > Alterar tipo de ambiente de execução > **GPU T4**. **2.** Rode cada célula com o botão ▶ (uma por vez).
Antes: no PC rode `node orquestra/treino.js preparar` e tenha em mãos `treino.jsonl` e `validacao.jsonl` (pasta `orquestra/casos`).

## 1. Conferir a placa

In [ ]:
!nvidia-smi

## 2. Instalar (leva alguns minutos)

In [ ]:
!pip install -q unsloth "trl>=0.20,<2" datasets

## 3. Enviar os dois arquivos
Vai abrir uma janela: escolha `treino.jsonl` e `validacao.jsonl`.

In [ ]:
from google.colab import files
enviados = files.upload()
assert 'treino.jsonl' in enviados and 'validacao.jsonl' in enviados, 'faltou um dos dois arquivos'
import json
n = sum(1 for _ in open('treino.jsonl', encoding='utf-8'))
print(n, 'exemplos de treino')
assert n >= 20, 'poucos exemplos: junte mais casos antes de treinar'
v = sum(1 for _ in open('validacao.jsonl', encoding='utf-8'))
print(v, 'exemplos de validação')
assert v >= 1, 'validacao.jsonl está vazio: sem ela não há nota antes/depois. Rode o preparar de novo com mais casos.'

## 4. Treinar (de 20 a 60 minutos)

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import inspect

def montar_trainer(SFTTrainer, SFTConfig, modelo, tok, treino, validacao, tamanho=2048):
    """Monta o treinador com os nomes certos da versão instalada do TRL (processing_class/max_length ou tokenizer/max_seq_length)."""
    cfg = inspect.signature(SFTConfig).parameters
    pars = inspect.signature(SFTTrainer.__init__).parameters
    config = SFTConfig(dataset_text_field='text', per_device_train_batch_size=2,
        gradient_accumulation_steps=4, num_train_epochs=2, learning_rate=2e-4,
        eval_strategy='epoch', output_dir='saida', report_to='none',
        **{('max_length' if 'max_length' in cfg else 'max_seq_length'): tamanho})
    return SFTTrainer(model=modelo, train_dataset=treino, eval_dataset=validacao, args=config,
        **{('processing_class' if 'processing_class' in pars else 'tokenizer'): tok})

BASE = 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit'
modelo, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=2048, load_in_4bit=True)
modelo = FastLanguageModel.get_peft_model(modelo, r=16, lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])

dados = load_dataset('json', data_files={'train': 'treino.jsonl', 'validation': 'validacao.jsonl'})
dados = dados.map(lambda ex: {'text': tok.apply_chat_template(ex['messages'], tokenize=False)})

montar_trainer(SFTTrainer, SFTConfig, modelo, tok, dados['train'], dados['validation']).train()

## 5. Gerar o arquivo do modelo e baixar

In [ ]:
modelo.save_pretrained_gguf('mixai-local-v2', tok, quantization_method='q4_k_m')
import glob
from google.colab import files
arq = glob.glob('mixai-local-v2*/*.gguf') + glob.glob('mixai-local-v2*.gguf')
print(arq)
files.download(arq[0])

Pronto. Volte ao Claude e diga que o arquivo `.gguf` foi baixado: ele monta o resto no PC e mede a nota com `orquestra/avaliar.js` antes de trocar o modelo.